# Web scraping allergen lists

The purpose of this notebook is to explore methods of extracting and saving allergens from web sources. The main sources of this info can be found here: https://www.contactdermatitisinstitute.com/database.php

## Example: Paraben Mix

In [1]:
# Import libraries
import requests
from bs4 import BeautifulSoup

In [2]:
URL = "https://www.contactdermatitisinstitute.com/paraben-mix-a.php"
r = requests.get(URL)
print(r.content)

b'\n\n\n<!DOCTYPE html>\n<html>\n\t<head lang="en">\n\t\t<link rel="stylesheet" type="text/css" href="https://cdnjs.cloudflare.com/ajax/libs/slick-carousel/1.8.1/slick.css">\n\t\t<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css">\n\n\n\t\t\n\t\t\n\t\t<meta charset="utf-8">\n\t\t<meta http-equiv="X-UA-Compatible" content="IE=edge">\n    \t<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">\n\t\t<meta name="robots" content="index, follow" />\n\t\t\n\t\t<meta name="description" content="The Contact Dermatitis Institute provides physicians and patients contact dermatitis research and information. Learn more.">\n\t\t<meta name="keywords" content="Contact Dermatitis Institute, CDI, contact dermatitis, dermatitis, contact dermatitis institute, contact dermatitis information, dermatitis research, contact dermatitis research, contact dermatitis treatments, contact dermatitis patient information">\n\t\

In [3]:
 # Create a BeautifulSoup object
soup = BeautifulSoup(r.content, 'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<html>
 <head lang="en">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/slick-carousel/1.8.1/slick.css" rel="stylesheet" type="text/css"/>
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css" rel="stylesheet"/>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <meta content="index, follow" name="robots">
   <meta content="The Contact Dermatitis Institute provides physicians and patients contact dermatitis research and information. Learn more." name="description"/>
   <meta content="Contact Dermatitis Institute, CDI, contact dermatitis, dermatitis, contact dermatitis institute, contact dermatitis information, dermatitis research, contact dermatitis research, contact dermatitis treatments, contact dermatitis patient information" name="keywords"/>
   <!-- Bootstrap CSS -->
   <link crossorigin="anonymou

In [4]:
# Navigate the data structure to extract the text
soup.find_all('p')

# or even better
soup.get_text()

"\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nParaben mix [A] | Allergic Contact Dermatitis Database\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\t\t          Training\n\t\t        \n\n2-Day In Person Training\n4-Hour Patch Test Insider E-Session\n4-Hour Patch Test E-Training\nOnline Training\n\n\n\n\n\t\t          Resources\n\t\t        \n\nPatient Forms\nPhysician Locator\nPublic Textbooks\n\n\n\nAllergen Database\n\n\n\nC.D.I. Community\n\n\n\n\n\n\n \n\n\n\n\n\n\nBACK TO SEARCH\nPRINT\n\n\n\nParaben mix [A]\n\nWhere is Paraben mix [A] found?\n\n\nParabens are used as preservatives in many over-the-counter medications,\ncosmetics, personal care, and hygiene products. Paraben mix contains the\nfollowing substances: Methyl p-hydroxybenzoate, Ethyl p-hydroxybenzoate,\nPropyl p-hydroxybenzoate, and Butyl p-hydroxybenzoate. Individuals allergic to\nparabens often experience reactions to these ingredients only on inflamed skin\nand may tolerate produc

We can find where the alternative ingredients names start because they always come after "Avoid products that list any of the following names in the ingredients:" and followed by "What are some products that may contain...". So if we can find the index for these could be as simple as just text[left:right]

In [5]:
text = soup.get_text()
print(text.index('ingredients:'))

1106


So this is the index of the first letter of our "ingredients:" above:

In [6]:
text[1106]

'i'

To get the right position as after this word we need to add the length of it to this number

In [7]:
print(text.index('ingredients:') + len('ingredients:'))

1118


Sorted. Now we need to find the index position for our right hand phrase after the ingredients list

In [8]:
print(text.index('What are some products that may contain'))

1536


So if we indexed text using this numbers we would get:

In [9]:
print(text[1118:1536])




• Methyl p-hydroxybenzoate
• Methylparaben
• 4-hydroxybenzoic acid methyl ester
• Methyl parahydroxybenzoate
• p-methoxycarbonylphenol
• Ethyl p-hydroxybenzoate
• Ethylparaben
• 4-hydroxybenzoic acid ethyl ester
• Ethyl p-oxybenzoate
• p-carbethoxyphenol
• Propyl p-hydroxybenzoate
• Propylparaben
• 4-hydroxybenzoic acid propyl ester
• Butyl p-hydroxybenzoate
• Butylparaben
• 4-hydroxybenzoic acid butyl ester






Belissimo. Fair bit of white space and weird bullet points but that shouldn't be an issue to clean. So translating the above in a reproducible way:

In [27]:
text = soup.get_text()
alt_allergen_names = text[text.index('ingredients:')+len('ingredients:'):text.index('What are some products that may contain')]
print(alt_allergen_names)




• Methyl p-hydroxybenzoate
• Methylparaben
• 4-hydroxybenzoic acid methyl ester
• Methyl parahydroxybenzoate
• p-methoxycarbonylphenol
• Ethyl p-hydroxybenzoate
• Ethylparaben
• 4-hydroxybenzoic acid ethyl ester
• Ethyl p-oxybenzoate
• p-carbethoxyphenol
• Propyl p-hydroxybenzoate
• Propylparaben
• 4-hydroxybenzoic acid propyl ester
• Butyl p-hydroxybenzoate
• Butylparaben
• 4-hydroxybenzoic acid butyl ester






Clean up this text. Found the strip() function was still leaving some '\n' characters so just replaced them

In [28]:
alt_allergen_names = alt_allergen_names.lower().strip()
alt_allergen_names = alt_allergen_names.replace('\n', '')
alt_allergen_names = alt_allergen_names.split('• ')
alt_allergen_names

['',
 'methyl p-hydroxybenzoate',
 'methylparaben',
 '4-hydroxybenzoic acid methyl ester',
 'methyl parahydroxybenzoate',
 'p-methoxycarbonylphenol',
 'ethyl p-hydroxybenzoate',
 'ethylparaben',
 '4-hydroxybenzoic acid ethyl ester',
 'ethyl p-oxybenzoate',
 'p-carbethoxyphenol',
 'propyl p-hydroxybenzoate',
 'propylparaben',
 '4-hydroxybenzoic acid propyl ester',
 'butyl p-hydroxybenzoate',
 'butylparaben',
 '4-hydroxybenzoic acid butyl ester']

In [29]:
type(alt_allergen_names)

list

Now we have a list we want to remove any empty elements left behind:

In [31]:
alt_allergen_names = [x for x in alt_allergen_names if x.strip()]
alt_allergen_names

['methyl p-hydroxybenzoate',
 'methylparaben',
 '4-hydroxybenzoic acid methyl ester',
 'methyl parahydroxybenzoate',
 'p-methoxycarbonylphenol',
 'ethyl p-hydroxybenzoate',
 'ethylparaben',
 '4-hydroxybenzoic acid ethyl ester',
 'ethyl p-oxybenzoate',
 'p-carbethoxyphenol',
 'propyl p-hydroxybenzoate',
 'propylparaben',
 '4-hydroxybenzoic acid propyl ester',
 'butyl p-hydroxybenzoate',
 'butylparaben',
 '4-hydroxybenzoic acid butyl ester']

Ultimately we want a dictionary with the main allergen as the key and then all the alternative names as values:

In [37]:
paraben_dict = {'Paraben Mix': alt_allergen_names}
print(paraben_dict['Paraben Mix'])

['methyl p-hydroxybenzoate', 'methylparaben', '4-hydroxybenzoic acid methyl ester', 'methyl parahydroxybenzoate', 'p-methoxycarbonylphenol', 'ethyl p-hydroxybenzoate', 'ethylparaben', '4-hydroxybenzoic acid ethyl ester', 'ethyl p-oxybenzoate', 'p-carbethoxyphenol', 'propyl p-hydroxybenzoate', 'propylparaben', '4-hydroxybenzoic acid propyl ester', 'butyl p-hydroxybenzoate', 'butylparaben', '4-hydroxybenzoic acid butyl ester']


## Web scrape to find list of all allergens and links to each page

Okay so fair enough we have a single example here but we need to find a way to scrape all the allergens from the website to collate.

In [ ]:
# Import libraries
import requests
from bs4 import BeautifulSoup

In [38]:
URL = "https://www.contactdermatitisinstitute.com/database.php"
r = requests.get(URL)
print(r.content)

b'<!DOCTYPE html>\n<html>\n\t<head lang="en">\n\t\t<link rel="stylesheet" type="text/css" href="https://cdnjs.cloudflare.com/ajax/libs/slick-carousel/1.8.1/slick.css">\n\t\t<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css">\n\n\n\t\t\n\t\t\n\t\t<meta charset="utf-8">\n\t\t<meta http-equiv="X-UA-Compatible" content="IE=edge">\n    \t<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">\n\t\t<meta name="robots" content="index, follow" />\n\t\t\n\t\t<meta name="description" content="The Contact Dermatitis Institute provides physicians and patients contact dermatitis research and information. Learn more.">\n\t\t<meta name="keywords" content="Contact Dermatitis Institute, CDI, contact dermatitis, dermatitis, contact dermatitis institute, contact dermatitis information, dermatitis research, contact dermatitis research, contact dermatitis treatments, contact dermatitis patient information">\n\t\t\n\t\

In [39]:
 # Create a BeautifulSoup object
soup = BeautifulSoup(r.content, 'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<html>
 <head lang="en">
  <link href="https://cdnjs.cloudflare.com/ajax/libs/slick-carousel/1.8.1/slick.css" rel="stylesheet" type="text/css"/>
  <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css" rel="stylesheet"/>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <meta content="index, follow" name="robots">
   <meta content="The Contact Dermatitis Institute provides physicians and patients contact dermatitis research and information. Learn more." name="description"/>
   <meta content="Contact Dermatitis Institute, CDI, contact dermatitis, dermatitis, contact dermatitis institute, contact dermatitis information, dermatitis research, contact dermatitis research, contact dermatitis treatments, contact dermatitis patient information" name="keywords"/>
   <!-- Bootstrap CSS -->
   <link crossorigin="anonymou

In [40]:
# Navigate the data structure to extract the text
soup.find_all('a')

[<a class="navbar-brand" href="doctors/index_doc.php"><img alt="Contact Dermatitis Institute™" id="cdi-logo2" src="patients/images/cdi-logo2.jpg" style="max-width:200px; "/></a>,
 <a aria-expanded="false" aria-haspopup="true" class="nav-link dropdown-toggle" data-toggle="dropdown" href="#" id="navbarDropdown" role="button">
 		          Training
 		        </a>,
 <a class="dropdown-item" href="doctors/two-day-patch-test-training.php">2-Day In Person Training</a>,
 <a class="dropdown-item" href="doctors/four-hour-patch-test-training-session.php">4-Hour Patch Test Insider E-Session</a>,
 <a class="dropdown-item" href="doctors/four-hour-patch-test-training.php">4-Hour Patch Test E-Training</a>,
 <a class="dropdown-item" href="doctors/online-training.php">Online Training</a>,
 <a aria-expanded="false" aria-haspopup="true" class="nav-link dropdown-toggle" data-toggle="dropdown" href="#" id="navbarDropdown" role="button">
 		          Resources
 		        </a>,
 <a class="dropdown-item" href

Let's be specific and go for item classes (includes "item sulfates" etc too)

In [42]:
soup.find_all('a', class_ = "item")

[<a class="item" href="nitrobutyl-morpholine-ethylnitro-trimethylene-dimorpholi.php" id="n"><img src="images/database-allergen.png"/>(nitrobutyl) morpholine / (ethylnitro-trimethylene) dimorpholi</a>,
 <a class="item" href="1-2-benzisothiazoline-3-one-sodium-salt.php" id="b"><img src="images/database-allergen.png"/>1,2-benzisothiazoline-3-one, sodium salt</a>,
 <a class="item" href="nitrobutyl-morpholine-ethylnitro-trimethylene-dimorpholi.php" id="b"><img src="images/database-allergen.png"/>1,3-butandiol-dimethacrylate</a>,
 <a class="item" href="1-3-diphenylguanidine.php" id="d"><img src="images/database-allergen.png"/>1,3-diphenylguanidine</a>,
 <a class="item" href="1-3-5-tris-2-hydroxyethyl-hexahydrotriazine-grotan-bk.php" id="t"><img src="images/database-allergen.png"/>1,3,5-tris-(2-hydroxyethyl)-hexahydrotriazine (grotan bk)</a>,
 <a class="item" href="1-4-butandioldimethacrylat-budma.php" id="b"><img src="images/database-allergen.png"/>1,4-butandioldimethacrylat (budma)</a>,
 <a

So this returns a list that we could iterate through to create a dictionary for each with just the item name and the relevant url:

In [57]:
database_list = soup.find_all('a', class_ = "item")
item = database_list[1]
item

<a class="item" href="1-2-benzisothiazoline-3-one-sodium-salt.php" id="b"><img src="images/database-allergen.png"/>1,2-benzisothiazoline-3-one, sodium salt</a>

We can extract the url for this item using a beautifulsoup function then it's just a case of prefixing it with https://www.contactdermatitisinstitute.com/ for the full item url

In [58]:
item.get("href")

'1-2-benzisothiazoline-3-one-sodium-salt.php'

In [59]:
# strip whitespace
item.get_text(strip=True)

'1,2-benzisothiazoline-3-one, sodium salt'

Above is actually a good example of one where the allergen item name is not clear and might be best taken from the "Avoid products that list any of the following names..." section in each page.

Either way we can create a function that extracts all of the urls into a list

In [61]:
database_list = soup.find_all('a', class_ = "item")

item_urls = []
base_url = "https://www.contactdermatitisinstitute.com/"

for item in database_list:
    href = item.get("href")

    full_url = base_url + href

    item_urls.append(full_url)

item_urls

['https://www.contactdermatitisinstitute.com/nitrobutyl-morpholine-ethylnitro-trimethylene-dimorpholi.php',
 'https://www.contactdermatitisinstitute.com/1-2-benzisothiazoline-3-one-sodium-salt.php',
 'https://www.contactdermatitisinstitute.com/nitrobutyl-morpholine-ethylnitro-trimethylene-dimorpholi.php',
 'https://www.contactdermatitisinstitute.com/1-3-diphenylguanidine.php',
 'https://www.contactdermatitisinstitute.com/1-3-5-tris-2-hydroxyethyl-hexahydrotriazine-grotan-bk.php',
 'https://www.contactdermatitisinstitute.com/1-4-butandioldimethacrylat-budma.php',
 'https://www.contactdermatitisinstitute.com/1-4-butanedioldiglycidyl-ether.php',
 'https://www.contactdermatitisinstitute.com/1-6-hexanediolediglycidyl-ether.php',
 'https://www.contactdermatitisinstitute.com/2-2-aminoethoxy-eth.php',
 'https://www.contactdermatitisinstitute.com/2-bromo-2nitropropane-1-3-diol-bronopol.php',
 'https://www.contactdermatitisinstitute.com/2-ethylhexyl-acrylate.php',
 'https://www.contactdermatitis

Now for each of these urls we want to follow the process as before to extract the ingredients lists in each

# Extracting allergen ingredients lists from all urls